In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf
import time

from sklearn import linear_model as lm

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

import re
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

import pandas as pd

## Load in the data from the first dataset

In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power'])


## Extract the relevant labels. Here I'm comparing behavior==1&cond==4 to behavior==2&cond=={4,6,8}

In [ ]:
myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
times = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

y = np.zeros(N)
y[indx_pos] = 1

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
times = times[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]


## Divide into the training and test sets

In [ ]:
N = len(mouse)

training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0

power = power*10
power[power>6] = 6

X = power[indx_tot]

X_train = X[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]


## Add in the newer dataset

In [ ]:
power,labels_new = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',
                             fBounds=(1,56),feature_list=['power'])
power = 10*power
power = power.astype(np.float32)
power[power>6] = 6

windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
times_new = np.squeeze(windows_new['time'])


## Divide this into training and test sets

In [ ]:
idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = power[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]


mice_new = np.unique(mouse_new)
nMice = len(mice_new)
mice_new_train = mice_new[:4]


ids = np.zeros(len(mouse_new))
for i in range(4):
    ids[mouse_new==mice_new_train[i]] = 1
    
X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]
m_train_new = mouse_new[ids==1]
m_test_new = mouse_new[ids==0]

In [ ]:
powerFeatures = labels['powerFeatures']
print(powerFeatures[::56])
regionKeys = ['IL','LHb','LSN','MDThal','MeA','NAc','OFC','PL','V1','VHipp','VMHvl']

# Combine the datasets

In [ ]:
X_train_total = np.vstack((X_train,X_train_new))
X_test_total = np.vstack((X_test,X_test_new))
y_train_total = np.concatenate((y_train,y_train_new))
y_test_total = np.concatenate((y_test,y_test_new))
m_train_total = np.concatenate((m_train,m_train_new))
m_test_total = np.concatenate((m_test,m_test_new))

## Now make some auxiliary methods

In [ ]:
# This method selects the covariates associated with a single region
def selectRegion(X_tr,X_te,regionKey): 
    nVariables = X_tr.shape[1]
    idxs = np.zeros(nVariables)
    for i in range(nVariables):
        if regionKey in powerFeatures[i]:
            idxs[i] = 1
    
    X_tr_sub = X_tr[:,idxs==1]
    X_te_sub = X_te[:,idxs==1]
    return X_tr_sub,X_te_sub,idxs

#This method evaluates the AUC on each of the holdout mice and returns values in a dictionary
def evaluateAUC(y_true,y_pred,m_indexes):
    mice_test = np.unique(m_indexes)
    out_dict = {}
    for i in range(len(mice_test)):
        y_true_sub = y_true[m_indexes==mice_test[i]]
        y_pred_sub = y_pred[m_indexes==mice_test[i]]
        if len(np.unique(y_true_sub)) == 2:
            out_dict[mice_test[i]] = roc_auc_score(y_true_sub,y_pred_sub)
        else:
            out_dict[mice_test[i]] = -1
        
    return out_dict

def returnAverage(out_dict):
    key_list = list(out_dict.keys())
    nKeys = len(key_list)
    out = np.zeros(nKeys)
    for i in range(nKeys):
        out[i] = out_dict[key_list[i]]
    out2 = out[out>=0]
    return np.mean(out2)

# Now lets actually test this out

### We will go step-by-step on how we examine the first region before going through the remaining regions in a loop

In [ ]:
Xtr_IL,Xte_IL,idx = selectRegion(X_train_total,X_test_total,regionKey=regionKeys[0])

In [ ]:
t1 = time.time()
model = lm.LogisticRegressionCV(solver='saga',Cs=np.logspace(-2,0,num=10),cv=3,penalty='l2',
                                max_iter=10000,n_jobs=10,
                                class_weight='balanced',
                                random_state=42)
model.fit(Xtr_IL,y_train_total)
t2 = time.time()
print('Runtime ',t2-t1,'Seconds')

In [ ]:
y_pred = model.decision_function(Xtr_IL)
out_dict = evaluateAUC(y_train_total,y_pred,m_train_total)
print('Training set average AUC',returnAverage(out_dict))

In [ ]:
y_pred = model.decision_function(Xte_IL)
out_dict = evaluateAUC(y_test_total,y_pred,m_test_total)
print('Testing set average AUC',returnAverage(out_dict))

## So basically a model trained on just the first region sucks. Now we loop through the entire set of regions

In [ ]:
nRegions = len(regionKeys)
trainingset_aucs = np.zeros(nRegions)
testingset_aucs = np.zeros(nRegions)
for i in range(nRegions):
    print(i)
    Xtr_sub,Xte_sub,idx = selectRegion(X_train_total,X_test_total,regionKey=regionKeys[i])
    model_reg = lm.LogisticRegressionCV(solver='saga',Cs=np.logspace(-2,0,num=10),cv=5,penalty='l2',
                                max_iter=10000,n_jobs=10,
                                class_weight='balanced',
                                random_state=42)
    model_reg.fit(Xtr_sub,y_train_total)
    
    y_pred = model_reg.decision_function(Xtr_sub)
    out_dict = evaluateAUC(y_train_total,y_pred,m_train_total)
    trainingset_aucs[i] = returnAverage(out_dict)
    
    y_pred = model_reg.decision_function(Xte_sub)
    out_dict = evaluateAUC(y_test_total,y_pred,m_test_total)    
    testingset_aucs[i] = returnAverage(out_dict)

In [ ]:
trainingset_aucs

In [ ]:
testingset_aucs

In [ ]:
df = pd.DataFrame({'Region':regionKeys,'TrainingSetAUC':trainingset_aucs,'TestingSetAUC':testingset_aucs})
df

In [ ]:
df.to_csv('Aggression_Single_Region_Prediction_12.csv',sep=',')

## Training mice

In [ ]:
np.unique(m_train_total)

## Testing mice

In [ ]:
np.unique(m_test_total)